# Krino Inference Server

Run Krino decision models on free Colab/Kaggle GPU and expose as a public API.

**How to use:**
1. Select GPU runtime: Runtime → Change runtime type → T4 GPU
2. Run all cells (Runtime → Run all)
3. Copy the public URL printed in Step 3 — available **immediately**, no model pre-loading needed
4. Paste into the [Krino playground](https://oaklight.github.io/krino/playground)
5. Select any model in the playground — it loads on first use

**Supports:** Colab (free T4) and Kaggle (free T4 x2, 30hrs/week)

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install torch transformers safetensors huggingface_hub gradio

# Install cloudflared for tunnel
import platform, subprocess, os
arch = platform.machine()
if arch == "x86_64":
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
elif arch == "aarch64":
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64"
else:
    raise RuntimeError(f"Unsupported architecture: {arch}")

subprocess.run(["wget", "-q", url, "-O", "/usr/local/bin/cloudflared"], check=True)
os.chmod("/usr/local/bin/cloudflared", 0o755)
print("✓ Dependencies installed")

## Step 2: Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✓ GPU: {gpu} ({vram:.1f} GB VRAM)")
else:
    print("⚠ No GPU detected — inference will be slow. Enable GPU runtime:")
    print("  Colab: Runtime → Change runtime type → T4 GPU")
    print("  Kaggle: Settings → Accelerator → GPU T4 x2")

## Step 3: Start Server + Tunnel

Starts the Gradio server and cloudflared tunnel immediately. Models are loaded **on first request** — no need to pre-select.

The public URL appears within seconds. All 6 Krino models are available via the model dropdown.

In [ ]:
import gradio as gr
import importlib.util, subprocess, threading, re, time, json
from huggingface_hub import hf_hub_download

AVAILABLE_MODELS = {
    "oaklight/krino-ettin-150m-heads": "Ettin-150m (fastest, 150M)",
    "oaklight/krino-modernbert-base-heads": "ModernBERT-base (149M)",
    "oaklight/krino-qwen3-0.6b-heads": "Qwen3-0.6B",
    "oaklight/krino-qwen3.5-4b-heads": "Qwen3.5-4B",
    "oaklight/krino-qwen3-reranker-4b-heads": "Qwen3-reranker-4B",
    "oaklight/krino-qwen3-reranker-0.6b-heads": "Qwen3-reranker-0.6B",
}

loaded_models = {}
loading_lock = threading.Lock()

def get_model(model_id):
    if model_id in loaded_models:
        return loaded_models[model_id]
    with loading_lock:
        if model_id in loaded_models:
            return loaded_models[model_id]
        print(f"Loading {model_id}...")
        t0 = time.time()
        path = hf_hub_download(model_id, "krino.py")
        spec = importlib.util.spec_from_file_location(
            f"krino_{model_id.replace('/', '_')}", path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        loaded_models[model_id] = mod.KrinoModel.from_pretrained(model_id)
        print(f"✓ {AVAILABLE_MODELS.get(model_id, model_id)} loaded in {time.time()-t0:.1f}s")
        return loaded_models[model_id]

def predict(model_id, state, question_type, instructions, options_text):
    question = {"type": question_type, "instructions": instructions}
    if question_type in ("choice", "score"):
        parsed = {}
        for line in (options_text or "").strip().splitlines():
            line = line.strip()
            if not line:
                continue
            if ":" in line:
                key, desc = line.split(":", 1)
                parsed[key.strip()] = desc.strip()
            else:
                parsed[line] = line
        if not parsed:
            return {"error": f"{question_type} requires at least one option (format: key: description)"}
        question["criteria" if question_type == "choice" else "legend"] = parsed

    try:
        m = get_model(model_id)
        t0 = time.perf_counter()
        answer = m.predict(state=state, question=question)
        answer["latency_ms"] = round((time.perf_counter() - t0) * 1000, 1)
        answer["model"] = AVAILABLE_MODELS.get(model_id, model_id)
        return answer
    except Exception as e:
        return {"error": str(e)}

def jev_proxy(api_key, payload_json):
    import urllib.request
    payload = payload_json.encode("utf-8")
    req = urllib.request.Request(
        "https://api.typesafe.ai/v1/systemone",
        data=payload,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    )
    try:
        resp = urllib.request.urlopen(req, timeout=30)
        return json.loads(resp.read())
    except Exception as e:
        return {"error": str(e)}

with gr.Blocks(title="Krino Inference Server") as demo:
    gr.Markdown("# Krino Inference Server\nModels load on first request (~15-30s).")
    with gr.Row():
        model_dd = gr.Dropdown(choices=list(AVAILABLE_MODELS.keys()),
            value="oaklight/krino-ettin-150m-heads", label="Model (loads on first use)")
        qtype_radio = gr.Radio(choices=["noul", "choice", "score"], value="choice", label="Question Type")
    state_tb = gr.Textbox(label="State / Input Text", lines=3)
    instr_tb = gr.Textbox(label="Instructions")
    opts_tb = gr.Textbox(label="Options (key: description, one per line)", lines=4)
    result_json = gr.JSON(label="Result")
    run_btn = gr.Button("Run")
    run_btn.click(predict, inputs=[model_dd, state_tb, qtype_radio, instr_tb, opts_tb], outputs=result_json, api_name="predict")

    # Hidden proxy endpoint for Jev API (CORS workaround)
    jev_key_tb = gr.Textbox(visible=False)
    jev_payload_tb = gr.Textbox(visible=False)
    jev_result = gr.JSON(visible=False)
    jev_btn = gr.Button(visible=False)
    jev_btn.click(jev_proxy, inputs=[jev_key_tb, jev_payload_tb], outputs=jev_result, api_name="jev_proxy")

    # Remote shutdown endpoint
    def shutdown():
        import os, signal, threading
        threading.Timer(1.0, lambda: os.kill(os.getpid(), signal.SIGINT)).start()
        return {"status": "shutting_down"}
    shutdown_result = gr.JSON(visible=False)
    shutdown_btn = gr.Button(visible=False)
    shutdown_btn.click(shutdown, inputs=[], outputs=shutdown_result, api_name="shutdown")

# Start Gradio in background
port = 7860
threading.Thread(
    target=lambda: demo.launch(server_port=port, share=False, quiet=True),
    daemon=True).start()
time.sleep(3)

# Start cloudflared tunnel
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
)

public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print(f"\n{'='*60}")
    print(f"  ✓ PUBLIC URL: {public_url}")
    print(f"{'='*60}")
    print(f"\nPaste this URL into the Krino playground:")
    print(f"  https://oaklight.github.io/krino/playground")
    print(f"\nModels load on first request (~15-30s for download + init).")
    print(f"Subsequent calls to the same model are fast (~30-300ms).")
else:
    print("⚠ Tunnel failed to start.")
    print("Fallback: use Gradio's share link instead.")

## Step 4: Keep Alive

Run this cell to keep the session active. Stop the cell (■) to shut down.

In [ ]:
import time
try:
    print("Server running. Press Stop (■) to shut down.")
    print("Model loading status will appear above when first requests arrive.")
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down...")
    tunnel_proc.terminate()
    print("✓ Server stopped")